## Link to the git 
https://github.com/mohankrishna02/interview-scenerios-spark-sql?tab=readme-ov-file

####Scenerio-1 Query to get who are getting equal salary

In [0]:
from pyspark.sql import Row

worker_data = [
    Row(workerid="001", firstname="Monika", lastname="Arora", salary=100000, joiningdate="2014-02-20 09:00:00", depart="HR"),
    Row(workerid="002", firstname="Niharika", lastname="Verma", salary=300000, joiningdate="2014-06-11 09:00:00", depart="Admin"),
    Row(workerid="003", firstname="Vishal", lastname="Singhal", salary=300000, joiningdate="2014-02-20 09:00:00", depart="HR"),
    Row(workerid="004", firstname="Amitabh", lastname="Singh", salary=500000, joiningdate="2014-02-20 09:00:00", depart="Admin"),
    Row(workerid="005", firstname="Vivek", lastname="Bhati", salary=500000, joiningdate="2014-06-11 09:00:00", depart="Admin"),
]

worker_df = spark.createDataFrame(worker_data)
worker_df.createOrReplaceTempView("worker")

display(worker_df)

workerid,firstname,lastname,salary,joiningdate,depart
001,Monika,Arora,100000,2014-02-20 09:00:00,HR
002,Niharika,Verma,300000,2014-06-11 09:00:00,Admin
003,Vishal,Singhal,300000,2014-02-20 09:00:00,HR
004,Amitabh,Singh,500000,2014-02-20 09:00:00,Admin
005,Vivek,Bhati,500000,2014-06-11 09:00:00,Admin


In [0]:
%sql
select a.*  from worker a inner join worker b on a.workerid<>b.workerid and a.salary=b.salary 

workerid,firstname,lastname,salary,joiningdate,depart
002,Niharika,Verma,300000,2014-06-11 09:00:00,Admin
003,Vishal,Singhal,300000,2014-02-20 09:00:00,HR
004,Amitabh,Singh,500000,2014-02-20 09:00:00,Admin
005,Vivek,Bhati,500000,2014-06-11 09:00:00,Admin


In [0]:
from pyspark.sql.functions import col

final_df = worker_df.alias("a").join(worker_df.alias("b"), (col("a.workerid") != col("b.workerid")) & (col("a.salary") == col("b.salary")), "inner").select("a.*")
display(final_df)

workerid,firstname,lastname,salary,joiningdate,depart
002,Niharika,Verma,300000,2014-06-11 09:00:00,Admin
003,Vishal,Singhal,300000,2014-02-20 09:00:00,HR
004,Amitabh,Singh,500000,2014-02-20 09:00:00,Admin
005,Vivek,Bhati,500000,2014-06-11 09:00:00,Admin


#### Scenerio-43 The business team wants to identify customers who registered on the platform but have never placed an order.You have two tables: customers and orders.

In [0]:
from pyspark.sql import Row

# --- Customers data ---
customers_data = [
    Row(customer_id=101, customer_name="John"),
    Row(customer_id=102, customer_name="David"),
    Row(customer_id=103, customer_name="Emma"),
    Row(customer_id=104, customer_name="Sara"),
    Row(customer_id=105, customer_name="Mike"),
    Row(customer_id=106, customer_name="Chris"),
]

customers_df = spark.createDataFrame(customers_data)
customers_df.createOrReplaceTempView("customers")

# --- Orders data ---
orders_data = [
    Row(order_id=1, customer_id=101, order_amount=500),
    Row(order_id=2, customer_id=102, order_amount=300),
    Row(order_id=3, customer_id=101, order_amount=700),
    Row(order_id=4, customer_id=104, order_amount=200),
    Row(order_id=5, customer_id=106, order_amount=900),
]

orders_df = spark.createDataFrame(orders_data)
orders_df.createOrReplaceTempView("orders")

display(customers_df)
display(orders_df)

customer_id,customer_name
101,John
102,David
103,Emma
104,Sara
105,Mike
106,Chris


order_id,customer_id,order_amount
1,101,500
2,102,300
3,101,700
4,104,200
5,106,900


In [0]:
%sql
select  *
from  customers c left join orders o
on c.customer_id = o.customer_id
where o.order_id is null;

customer_id,customer_name,order_id,customer_id,order_amount
103,Emma,null,null,null
105,Mike,null,null,null


In [0]:
df_cust_no_order = customers_df.join(orders_df, "customer_id", "leftanti")
display(df_cust_no_order)

customer_id,customer_name
103,Emma
105,Mike
